### KNeighbors Classifier

In [7]:
import pandas as pd
import time
import warnings
import tracemalloc
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score
from scipy.stats import loguniform

In [8]:
def kneighbors(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    knn = KNeighborsClassifier()

    tracemalloc.start()
    start_time = time.time()

    knn.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    y_pred = knn.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb

#### Mетрики без подбора гиперпараметров

In [9]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage = kneighbors(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.911
Среднее время = 0.00153 сек
Среднее потребление памяти = 0.172 MB


,samples (n),features (m),f1-score,time (sec),memory (MB)
0,100,5,1.000000,0.002953,0.016376
1,100,8,1.000000,0.001446,0.017978
2,100,11,0.814815,0.001411,0.030437
3,500,5,0.993377,0.001535,0.052424
4,500,8,0.951613,0.001302,0.073932
5,500,11,0.769231,0.001200,0.117175
6,1000,5,0.985294,0.001285,0.098414
7,1000,8,0.985294,0.001182,0.140285
8,1000,11,0.722022,0.001269,0.216081
9,3000,5,0.998770,0.001401,0.271771


Метод ближайших соседей показывает `f1-score` меньший, чем в Linear SVC и Native Bayes.

#### Mетрики с подбором гиперпараметров

In [10]:
def kneighbors_params(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    param_dist = {
        'n_neighbors': list(range(1, 31)),
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }

    knn = KNeighborsClassifier()

    random_search = RandomizedSearchCV(
        knn, param_dist, n_iter=10, cv=5, scoring='f1', random_state=81
    )

    tracemalloc.start()
    start_time = time.time()

    random_search.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    best_params = random_search.best_params_
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb, best_params, best_model

In [11]:
results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage, best_p, model = kneighbors_params(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage,
            'n_neighbors': best_p['n_neighbors'],
            'weights': best_p['weights']
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.922
Среднее время = 0.60217 сек
Среднее потребление памяти = 0.653 MB


,samples (n),features (m),f1-score,time (sec),memory (MB),n_neighbors,weights
0,100,5,1.000000,0.297435,0.231363,20,uniform
1,100,8,1.000000,0.327866,0.218693,6,distance
2,100,11,0.814815,0.315681,0.246224,19,distance
3,500,5,0.993377,0.432395,0.335755,6,distance
4,500,8,0.952381,0.382438,0.351310,5,distance
5,500,11,0.787402,0.413595,0.413116,6,distance
6,1000,5,0.996364,0.509833,0.515258,6,distance
7,1000,8,0.981685,0.544558,0.550879,14,uniform
8,1000,11,0.765799,0.597275,0.690316,6,distance
9,3000,5,0.998773,1.020718,1.199535,5,distance


После подбора гиперпараметра `f1-score` увеличился.

Диапазон `n_neighbors` от 1 до 30 выбран для поиска оптимального баланса между переобучением и избыточным обобщением данных.

Сравнение типов весов (`weights`) позволило определить, критична ли для прогноза аварии степень близости соседа.

Использование двух различных метрик расстояния (`euclidean` и `manhattan`) обусловлено числом присутствующих признаков: это позволило выбрать метод измерения, наиболее устойчивый к особенностям распределения данных в датасете.

### Сохранение модели

In [14]:
import joblib

joblib.dump(model, 'models/knn_model.pkl')

['models/knn_model.pkl']